In [ ]:


import duckdb
import pandas as pd

con = duckdb.connect('../Data/processed/project.duckdb')

In [2]:
#: How much data do we actually have?

con.execute("""
    SELECT COUNT(*) AS total_trades,
           COUNT(DISTINCT accountId) AS total_accounts,
           COUNT(DISTINCT campaignId) AS total_campaigns
    FROM trades
""").fetchdf()

,total_trades,total_accounts,total_campaigns
0,46520,502,34


a note here - in the provided datasets, we dont have campaigns 1-66 but 33-66 lolol 

In [ ]:
#Rows per campaign

con.execute("""
    SELECT campaignId, COUNT(*) AS num_trades
    FROM trades
    GROUP BY campaignId
    ORDER BY campaignId
""").fetchdf()

,campaignId,num_trades
0,33,749
1,34,942
2,35,1030
3,36,1457
4,37,1432
5,38,1271
6,39,1180
7,40,1432
8,41,1227
9,42,1674


In [3]:
# Trades per account 

con.execute("""
    SELECT accountId, COUNT(*) AS num_trades
    FROM trades
    GROUP BY accountId
    ORDER BY num_trades DESC
""").fetchdf()

,accountId,num_trades
0,D#1702482,228
1,D#1702476,208
2,D#1702452,202
3,D#1702588,193
4,D#1702696,193
...,...,...
497,D#1702712,31
498,D#1702384,30
499,D#1645625,7
500,D#1645639,2


In [ ]:
# avg trades per account
con.execute("""
    WITH trade_counts AS (
        SELECT accountId, COUNT(*) AS num_trades
        FROM trades
        GROUP BY accountId
    )
    SELECT 
        AVG(num_trades) AS avg_trades_per_account,
        MEDIAN(num_trades) AS median_trades_per_account,
        MODE(num_trades) AS mode_trades_per_account
    FROM trade_counts
""").fetchdf()

,avg_trades_per_account,median_trades_per_account,mode_trades_per_account
0,92.669323,90.0,71


In [5]:
#distribution check

con.execute("""
    SELECT 
        MIN(netProfit) AS min_profit, MAX(netProfit) AS max_profit, AVG(netProfit) AS avg_profit,
        MIN(durationSec) AS min_duration, MAX(durationSec) AS max_duration, AVG(durationSec) AS avg_duration,
        MIN(amount) AS min_lot, MAX(amount) AS max_lot, AVG(amount) AS avg_lot
    FROM trades
""").fetchdf()

,min_profit,max_profit,avg_profit,min_duration,max_duration,avg_duration,min_lot,max_lot,avg_lot
0,-944.1,680.64,-6.686877,0,79210,1415.867261,0.01,0.63,0.18378


In [7]:
#missing values 

con.execute("""
    SELECT 
        SUM(CASE WHEN slPrice IS NULL THEN 1 ELSE 0 END) AS no_sl_count,
        SUM(CASE WHEN tpPrice IS NULL THEN 1 ELSE 0 END) AS no_tp_count,
        COUNT(*) AS total
    FROM trades
""").fetchdf()

,no_sl_count,no_tp_count,total
0,24601.0,20930.0,46520


This is actually a really meaningful result — let's break it down.
What it's saying
Out of 46,520 total trades:

24,601 trades (≈53%) had no stop-loss set
20,930 trades (≈45%) had no take-profit set

Why this matters
Remember — this isn't "missing data" to clean away, it's a behavior signal in disguise. Over half your trades were placed with no stop-loss at all. That's a genuinely notable finding on its own, even before you build anything fancier:

No-SL trading is a classic risk-blindness/overconfidence pattern
You could directly test: do accounts with a higher no-SL rate breach the drawdown/consistency rules more often, or perform worse overall?
This alone could be one of your first real hypotheses for the memo: "Traders who don't set stop-losses are more likely to hard-breach or underperform vs. traders who consistently use them."

In [8]:
#Is the no-SL behavior spread evenly across traders, or concentrated in a subset?

con.execute("""
    SELECT accountId,
           COUNT(*) AS total_trades,
           SUM(CASE WHEN slPrice IS NULL THEN 1 ELSE 0 END) AS no_sl_trades,
           ROUND(SUM(CASE WHEN slPrice IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS no_sl_pct
    FROM trades
    GROUP BY accountId
    HAVING COUNT(*) >= 10
    ORDER BY no_sl_pct DESC
""").fetchdf()

,accountId,total_trades,no_sl_trades,no_sl_pct
0,D#1702501,94,87.0,92.6
1,D#1670967,100,92.0,92.0
2,D#1702476,208,190.0,91.3
3,D#1702471,110,95.0,86.4
4,D#1702503,79,68.0,86.1
...,...,...,...,...
494,D#1702667,65,15.0,23.1
495,D#1670918,101,23.0,22.8
496,D#1702457,155,35.0,22.6
497,D#1671059,88,17.0,19.3


we should explore this more.... 
we can export this output to a csv file, put it on tablue and check it out 
